In [101]:
import pandas as pd
import numpy as np

carbon=pd.read_csv('carbon_logs.csv',dtype={
        "CarbonLogID": "str",
        "LegID": "str",
        "CarrierID": "str",
        "ReportedMethod": "str"
    })
telemetry=pd.read_csv('telemetry.csv',dtype={
        "TelemetryID": "str",
        "ShipmentID": "str",
        "PowerSource": "str",
        "PingStatus": "str"
    })

In [102]:
carbon.info()
carbon.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 27031 entries, 0 to 27030
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   CarbonLogID           27031 non-null  str    
 1   LegID                 27031 non-null  str    
 2   CarrierID             27031 non-null  str    
 3   ReportedCO2_KG        27031 non-null  float64
 4   FuelConsumed_Gallons  27031 non-null  float64
 5   TrueCO2_KG            27031 non-null  float64
 6   ReportedMethod        27031 non-null  str    
dtypes: float64(3), str(4)
memory usage: 2.9 MB


,CarbonLogID,LegID,CarrierID,ReportedCO2_KG,FuelConsumed_Gallons,TrueCO2_KG,ReportedMethod
0,CAR-LEG-SHP-100001-1,LEG-SHP-100001-1,C10,10.79,262.50,2672.27,Fuel-Flow-Meter
1,CAR-LEG-SHP-100001-2,LEG-SHP-100001-2,C06,12.13,123.71,1259.32,Fuel-Flow-Meter
2,CAR-LEG-SHP-100001-3,LEG-SHP-100001-3,C08,30.53,81.91,833.83,Fuel-Flow-Meter
3,CAR-LEG-SHP-100002-1,LEG-SHP-100002-1,C07,87.98,41.53,422.81,Fuel-Flow-Meter
4,CAR-LEG-SHP-100003-1,LEG-SHP-100003-1,C10,22.30,538.71,5484.05,Fuel-Flow-Meter
5,CAR-LEG-SHP-100003-2,LEG-SHP-100003-2,C06,25.01,253.76,2583.31,Fuel-Flow-Meter
6,CAR-LEG-SHP-100003-3,LEG-SHP-100003-3,C09,61.78,128.22,1305.32,Fuel-Flow-Meter
7,CAR-LEG-SHP-100004-1,LEG-SHP-100004-1,C01,122.11,2470.94,25154.12,Fuel-Flow-Meter
8,CAR-LEG-SHP-100005-1,LEG-SHP-100005-1,C10,43.58,1047.59,10664.49,Fuel-Flow-Meter
9,CAR-LEG-SHP-100005-2,LEG-SHP-100005-2,C09,164.02,357.84,3642.82,Fuel-Flow-Meter


In [103]:
telemetry.info()
telemetry.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 83086 entries, 0 to 83085
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   TelemetryID    83086 non-null  str    
 1   ShipmentID     83086 non-null  str    
 2   Timestamp      83086 non-null  str    
 3   Latitude       83086 non-null  float64
 4   Longitude      83086 non-null  float64
 5   Temperature_C  83086 non-null  float64
 6   PowerSource    83086 non-null  str    
 7   PingStatus     83086 non-null  str    
dtypes: float64(3), str(5)
memory usage: 9.5 MB


,TelemetryID,ShipmentID,Timestamp,Latitude,Longitude,Temperature_C,PowerSource,PingStatus
0,TEL-SHP-100002-0,SHP-100002,2024-04-14 17:00:00,20.0,110.0,3.2,Main,Active
1,TEL-SHP-100002-1,SHP-100002,2024-04-15 17:00:00,21.2,107.9,3.6,Main,Active
2,TEL-SHP-100002-2,SHP-100002,2024-04-16 17:00:00,22.4,105.8,4.3,Main,Active
3,TEL-SHP-100005-0,SHP-100005,2025-07-29 00:00:00,20.0,110.0,4.7,Main,Active
4,TEL-SHP-100005-1,SHP-100005,2025-07-30 00:00:00,21.2,107.9,3.3,Main,Active
5,TEL-SHP-100005-2,SHP-100005,2025-07-31 00:00:00,22.4,105.8,3.3,Main,Active
6,TEL-SHP-100005-3,SHP-100005,2025-08-01 00:00:00,23.6,103.7,4.4,Main,Active
7,TEL-SHP-100005-4,SHP-100005,2025-08-02 00:00:00,24.8,101.6,4.2,Main,Active
8,TEL-SHP-100005-5,SHP-100005,2025-08-03 00:00:00,26.0,99.5,3.8,Main,Active
9,TEL-SHP-100005-6,SHP-100005,2025-08-04 00:00:00,27.2,97.4,4.2,Main,Active


In [104]:
text_columns = [
    "CarbonLogID",
    "LegID",
    "CarrierID",
    "ReportedMethod"
]

for col in text_columns:
    carbon[col] = carbon[col].astype(str).str.strip()

In [105]:
numeric_columns = [
    "ReportedCO2_KG",
    "FuelConsumed_Gallons",
    "TrueCO2_KG"
]

for col in numeric_columns:
    carbon[col] = pd.to_numeric(carbon[col], errors="coerce")

In [106]:
carbon["DuplicateCarbonLogFlag"] = carbon["CarbonLogID"].duplicated()

carbon = carbon.drop_duplicates(subset="CarbonLogID", keep="first")

In [107]:
carbon["NegativeFuelFlag"] = carbon["FuelConsumed_Gallons"] < 0
carbon["NegativeReportedCO2Flag"] = carbon["ReportedCO2_KG"] < 0
carbon["NegativeTrueCO2Flag"] = carbon["TrueCO2_KG"] < 0

carbon["MissingFuelFlag"] = carbon["FuelConsumed_Gallons"].isna()
carbon["MissingReportedCO2Flag"] = carbon["ReportedCO2_KG"].isna()
carbon["MissingTrueCO2Flag"] = carbon["TrueCO2_KG"].isna()

carbon["ZeroFuelFlag"] = carbon["FuelConsumed_Gallons"] == 0

In [108]:
# --------------------------------------------------
# CO2 gap
# --------------------------------------------------
carbon["CO2Gap_KG"] = carbon["TrueCO2_KG"] - carbon["ReportedCO2_KG"]

carbon["CO2GapPct"] = np.where(
    carbon["TrueCO2_KG"] > 0,
    carbon["CO2Gap_KG"] / carbon["TrueCO2_KG"],
    np.nan
)

# --------------------------------------------------
# Under / over reporting flags
# --------------------------------------------------
carbon["CO2UnderReportedFlag"] = carbon["CO2GapPct"] > 0.05
carbon["CO2OverReportedFlag"] = carbon["CO2GapPct"] < -0.05

In [109]:
carbon["ImpliedCO2PerGallon"] = np.where(
    carbon["FuelConsumed_Gallons"] > 0,
    carbon["TrueCO2_KG"] / carbon["FuelConsumed_Gallons"],
    np.nan
)

In [110]:
legs=pd.read_csv(r'C:\Users\nooru\Desktop\logistics_dashboard\final_clean_data\shipment_leg_cleaned_final.csv')

In [111]:
# --------------------------------------------------
# Get leg information
# --------------------------------------------------
legs_info = legs[[
    "LegID",
    "ShipmentID",
    "CarrierID",
    "Mode",
    "DepartureTime"
]].copy()

legs_info = legs_info.rename(columns={
    "CarrierID": "LegCarrierID",
    "Mode": "LegMode",
    "DepartureTime": "LegDepartureTime"
})

# --------------------------------------------------
# Merge carbon with legs
# --------------------------------------------------
carbon = carbon.merge(
    legs_info,
    on="LegID",
    how="left"
)

# --------------------------------------------------
# Validation flags
# --------------------------------------------------
carbon["MissingLegFlag"] = carbon["ShipmentID"].isna()

carbon["CarbonCarrierMismatchFlag"] = (
    carbon["LegCarrierID"].notna() &
    (carbon["CarrierID"] != carbon["LegCarrierID"])
)

In [112]:
# --------------------------------------------------
# Aggregate carbon by shipment
# --------------------------------------------------
carbon_valid = carbon[carbon["MissingLegFlag"] == False].copy()

carbon_shipment_agg = carbon_valid.groupby("ShipmentID").agg(
    CarbonLogCount=("CarbonLogID", "count"),
    TotalFuelConsumed_Gallons=("FuelConsumed_Gallons", "sum"),
    TotalReportedCO2_KG=("ReportedCO2_KG", "sum"),
    TotalTrueCO2_KG=("TrueCO2_KG", "sum"),
    CO2UnderReportedLegCount=("CO2UnderReportedFlag", "sum"),
    CO2OverReportedLegCount=("CO2OverReportedFlag", "sum"),
    CarbonCarrierMismatchCount=("CarbonCarrierMismatchFlag", "sum")
).reset_index()

# --------------------------------------------------
# Shipment-level CO2 gap
# --------------------------------------------------
carbon_shipment_agg["CO2Gap_KG"] = (
    carbon_shipment_agg["TotalTrueCO2_KG"] -
    carbon_shipment_agg["TotalReportedCO2_KG"]
)

carbon_shipment_agg["CO2GapPct"] = np.where(
    carbon_shipment_agg["TotalTrueCO2_KG"] > 0,
    carbon_shipment_agg["CO2Gap_KG"] / carbon_shipment_agg["TotalTrueCO2_KG"],
    np.nan
)

carbon_shipment_agg["CO2UnderReportedShipmentFlag"] = (
    carbon_shipment_agg["CO2GapPct"] > 0.05
)

carbon_shipment_agg["CO2OverReportedShipmentFlag"] = (
    carbon_shipment_agg["CO2GapPct"] < -0.05
)

In [113]:

# --------------------------------------------------
# Trim text columns
# --------------------------------------------------
text_columns = [
    "TelemetryID",
    "ShipmentID",
    "PowerSource",
    "PingStatus"
]

for col in text_columns:
    telemetry[col] = telemetry[col].astype(str).str.strip()

# --------------------------------------------------
# Convert timestamp
# --------------------------------------------------
telemetry["Timestamp"] = (
    pd.to_datetime(telemetry["Timestamp"], errors="coerce")
    .astype("datetime64[ns]")
)

# --------------------------------------------------
# Convert numeric columns
# --------------------------------------------------
telemetry["Latitude"] = pd.to_numeric(telemetry["Latitude"], errors="coerce")
telemetry["Longitude"] = pd.to_numeric(telemetry["Longitude"], errors="coerce")
telemetry["Temperature_C"] = pd.to_numeric(telemetry["Temperature_C"], errors="coerce")

In [114]:
# --------------------------------------------------
# Duplicate checks
# --------------------------------------------------
telemetry["DuplicateTelemetryIDFlag"] = telemetry["TelemetryID"].duplicated()

telemetry = telemetry.drop_duplicates(subset="TelemetryID", keep="first")

# Remove exact duplicate shipment timestamp events
telemetry = telemetry.drop_duplicates(
    subset=["ShipmentID", "Timestamp"],
    keep="first"
)

# --------------------------------------------------
# Missing timestamp flag
# --------------------------------------------------
telemetry["MissingTimestampFlag"] = telemetry["Timestamp"].isna()

# --------------------------------------------------
# Future telemetry flag
# --------------------------------------------------
today = pd.Timestamp.today().normalize()

telemetry["FutureTelemetryFlag"] = telemetry["Timestamp"] > today

In [115]:
telemetry["InvalidLatitudeFlag"] = (
    (telemetry["Latitude"] < -90) |
    (telemetry["Latitude"] > 90) |
    telemetry["Latitude"].isna()
)

telemetry["InvalidLongitudeFlag"] = (
    (telemetry["Longitude"] < -180) |
    (telemetry["Longitude"] > 180) |
    telemetry["Longitude"].isna()
)

telemetry["InvalidLocationFlag"] = (
    telemetry["InvalidLatitudeFlag"] |
    telemetry["InvalidLongitudeFlag"]
)

In [116]:
shipment_clean=pd.read_csv(r'C:\Users\nooru\Desktop\logistics_dashboard\final_clean_data\shipment_cleaned_final.csv')

In [117]:
# --------------------------------------------------
# Valid shipment check
# --------------------------------------------------
valid_shipments = set(shipment_clean["ShipmentID"])

telemetry["OrphanShipmentFlag"] = ~telemetry["ShipmentID"].isin(valid_shipments)

# --------------------------------------------------
# Merge shipment storage requirement
# --------------------------------------------------
telemetry = telemetry.merge(
    shipment_clean[[
        "ShipmentID",
        "StorageRequirement",
        "CargoType",
        "BookingDate",
        "ActualDeliveryDate"
    ]],
    on="ShipmentID",
    how="left"
)

In [118]:
# --------------------------------------------------
# Temperature excursion flags
# --------------------------------------------------
telemetry["TempExcursionFlag"] = False
telemetry["FrozenCargoTempExcursionFlag"] = False

# Storage requirement groups
refrigerated = telemetry["StorageRequirement"].str.contains(
    "refrigerated",
    case=False,
    na=False
)

frozen_storage = telemetry["StorageRequirement"].str.contains(
    "frozen",
    case=False,
    na=False
)

dry_storage = telemetry["StorageRequirement"].str.contains(
    "dry",
    case=False,
    na=False
)

# Cargo-based frozen check
frozen_cargo = telemetry["CargoType"].str.contains(
    "frozen",
    case=False,
    na=False
)

# Refrigerated threshold: usually 2°C to 8°C
telemetry.loc[
    refrigerated &
    (
        (telemetry["Temperature_C"] < 2) |
        (telemetry["Temperature_C"] > 8)
    ),
    "TempExcursionFlag"
] = True

# Frozen storage threshold: usually below -18°C
telemetry.loc[
    frozen_storage &
    (telemetry["Temperature_C"] > -18),
    "TempExcursionFlag"
] = True

# Dry cargo threshold: usually 10°C to 30°C
telemetry.loc[
    dry_storage &
    (
        (telemetry["Temperature_C"] < 10) |
        (telemetry["Temperature_C"] > 30)
    ),
    "TempExcursionFlag"
] = True

# Special check: frozen cargo but temperature above -18°C
telemetry.loc[
    frozen_cargo &
    (telemetry["Temperature_C"] > -18),
    "FrozenCargoTempExcursionFlag"
] = True

In [119]:
# --------------------------------------------------
# Power source
# --------------------------------------------------
telemetry["PowerSource"] = telemetry["PowerSource"].str.strip().str.title()

telemetry["IsBatteryPower"] = telemetry["PowerSource"].str.lower().isin([
    "battery",
    "backup",
    "battery backup"
])

# --------------------------------------------------
# Ping status
# --------------------------------------------------
telemetry["PingStatus"] = telemetry["PingStatus"].str.strip().str.title()

telemetry["IsActivePing"] = telemetry["PingStatus"].str.lower() == "active"
telemetry["IsInactivePing"] = ~telemetry["IsActivePing"]

In [120]:
# --------------------------------------------------
# 1. Force convert date columns to datetime
# --------------------------------------------------
date_columns = ["Timestamp", "BookingDate", "ActualDeliveryDate"]

for col in date_columns:
    if col in telemetry.columns:
        telemetry[col] = pd.to_datetime(telemetry[col], errors="coerce")

# --------------------------------------------------
# 2. Now create the date flags safely
# --------------------------------------------------
telemetry["TelemetryDate"] = telemetry["Timestamp"].dt.normalize()

telemetry["TelemetryBeforeBookingFlag"] = (
    telemetry["Timestamp"].notna() &
    telemetry["BookingDate"].notna() &
    (telemetry["Timestamp"] < telemetry["BookingDate"])
)

telemetry["TelemetryAfterDeliveryFlag"] = (
    telemetry["Timestamp"].notna() &
    telemetry["ActualDeliveryDate"].notna() &
    (telemetry["Timestamp"] > (telemetry["ActualDeliveryDate"] + pd.Timedelta(days=1)))
)

In [121]:
telemetry_valid = telemetry[telemetry["OrphanShipmentFlag"] == False].copy()

telemetry_shipment_agg = telemetry_valid.groupby("ShipmentID").agg(
    TelemetryEventCount=("TelemetryID", "count"),
    FirstTelemetryTime=("Timestamp", "min"),
    LastTelemetryTime=("Timestamp", "max"),
    ActivePingCount=("IsActivePing", "sum"),
    InactivePingCount=("IsInactivePing", "sum"),
    BatteryPowerCount=("IsBatteryPower", "sum"),
    AvgTemperature_C=("Temperature_C", "mean"),
    MinTemperature_C=("Temperature_C", "min"),
    MaxTemperature_C=("Temperature_C", "max"),
    TempExcursionCount=("TempExcursionFlag", "sum"),
    FrozenCargoTempExcursionCount=("FrozenCargoTempExcursionFlag", "sum"),
    InvalidLocationCount=("InvalidLocationFlag", "sum"),
    FutureTelemetryCount=("FutureTelemetryFlag", "sum"),
    TelemetryBeforeBookingCount=("TelemetryBeforeBookingFlag", "sum"),
    TelemetryAfterDeliveryCount=("TelemetryAfterDeliveryFlag", "sum")
).reset_index()

In [122]:
telemetry_shipment_agg["ActivePingPct"] = np.where(
    telemetry_shipment_agg["TelemetryEventCount"] > 0,
    telemetry_shipment_agg["ActivePingCount"] / telemetry_shipment_agg["TelemetryEventCount"],
    0
)

telemetry_shipment_agg["BatteryPowerPct"] = np.where(
    telemetry_shipment_agg["TelemetryEventCount"] > 0,
    telemetry_shipment_agg["BatteryPowerCount"] / telemetry_shipment_agg["TelemetryEventCount"],
    0
)

telemetry_shipment_agg["TempExcursionFlag"] = (
    telemetry_shipment_agg["TempExcursionCount"] > 0
)

telemetry_shipment_agg["FrozenCargoTempExcursionFlag"] = (
    telemetry_shipment_agg["FrozenCargoTempExcursionCount"] > 0
)

telemetry_shipment_agg["LowActivePingFlag"] = (
    telemetry_shipment_agg["ActivePingPct"] < 0.80
)

In [123]:
telemetry_daily_agg = telemetry_valid.groupby([
    "ShipmentID",
    "TelemetryDate"
]).agg(
    TelemetryEventCount=("TelemetryID", "count"),
    ActivePingCount=("IsActivePing", "sum"),
    InactivePingCount=("IsInactivePing", "sum"),
    BatteryPowerCount=("IsBatteryPower", "sum"),
    AvgTemperature_C=("Temperature_C", "mean"),
    MinTemperature_C=("Temperature_C", "min"),
    MaxTemperature_C=("Temperature_C", "max"),
    TempExcursionCount=("TempExcursionFlag", "sum"),
    FrozenCargoTempExcursionCount=("FrozenCargoTempExcursionFlag", "sum"),
    InvalidLocationCount=("InvalidLocationFlag", "sum")
).reset_index()

In [124]:
last_location = (
    telemetry_valid.sort_values("Timestamp")
    .groupby("ShipmentID")[[
        "Latitude",
        "Longitude",
        "Timestamp"
    ]]
    .last()
    .reset_index()
)

last_location = last_location.rename(columns={
    "Latitude": "LastLatitude",
    "Longitude": "LastLongitude",
    "Timestamp": "LastTelemetryTimestamp"
})

telemetry_shipment_agg = telemetry_shipment_agg.merge(
    last_location,
    on="ShipmentID",
    how="left"
)

In [125]:
def safe_merge(base, new_table, key):
    """
    Merges new_table into base table.
    Removes duplicate columns from base before merging.
    """
    duplicate_columns = [
        col for col in new_table.columns
        if col != key and col in base.columns
    ]

    base = base.drop(columns=duplicate_columns, errors="ignore")

    return base.merge(
        new_table,
        on=key,
        how="left"
    )

In [126]:
shipment_final = shipment_clean.copy()

# Merge carbon aggregation
shipment_final = safe_merge(
    shipment_final,
    carbon_shipment_agg,
    "ShipmentID"
)

# Merge telemetry aggregation
shipment_final = safe_merge(
    shipment_final,
    telemetry_shipment_agg,
    "ShipmentID"
)

In [127]:
# --------------------------------------------------
# Carbon availability flags
# --------------------------------------------------
shipment_final["HasCarbonDataFlag"] = shipment_final["TotalTrueCO2_KG"].notna()
shipment_final["NoCarbonLogFlag"] = ~shipment_final["HasCarbonDataFlag"]

# --------------------------------------------------
# Telemetry availability flags
# --------------------------------------------------
shipment_final["TelemetryEventCount"] = shipment_final["TelemetryEventCount"].fillna(0)
shipment_final["HasTelemetryFlag"] = shipment_final["TelemetryEventCount"] > 0
shipment_final["NoTelemetryFlag"] = ~shipment_final["HasTelemetryFlag"]

# --------------------------------------------------
# Cold cargo without telemetry
# --------------------------------------------------
shipment_final["ColdCargoFlag"] = shipment_final["StorageRequirement"].str.contains(
    "Refrigerated|Frozen",
    case=False,
    na=False
)

shipment_final["ColdCargoNoTelemetryFlag"] = (
    shipment_final["ColdCargoFlag"] &
    shipment_final["NoTelemetryFlag"]
)

In [128]:
# --------------------------------------------------
# CO2 per ton
# --------------------------------------------------
shipment_final["TrueCO2PerTon"] = np.where(
    (shipment_final["TotalWeight_KG"] > 0) &
    (shipment_final["TotalTrueCO2_KG"].notna()),
    shipment_final["TotalTrueCO2_KG"] / (shipment_final["TotalWeight_KG"] / 1000),
    0
)

shipment_final["ReportedCO2PerTon"] = np.where(
    (shipment_final["TotalWeight_KG"] > 0) &
    (shipment_final["TotalReportedCO2_KG"].notna()),
    shipment_final["TotalReportedCO2_KG"] / (shipment_final["TotalWeight_KG"] / 1000),
    0
)

# --------------------------------------------------
# CO2 per USD declared value
# --------------------------------------------------
shipment_final["TrueCO2Per1000USD"] = np.where(
    (shipment_final["DeclaredValue_USD"] > 0) &
    (shipment_final["TotalTrueCO2_KG"].notna()),
    shipment_final["TotalTrueCO2_KG"] / (shipment_final["DeclaredValue_USD"] / 1000),
    0
)

In [129]:
# Carbon numeric columns
carbon_numeric_columns = [
    "CarbonLogCount",
    "TotalFuelConsumed_Gallons",
    "TotalReportedCO2_KG",
    "TotalTrueCO2_KG",
    "CO2UnderReportedLegCount",
    "CO2OverReportedLegCount",
    "CarbonCarrierMismatchCount",
    "CO2Gap_KG"
]

for col in carbon_numeric_columns:
    if col in shipment_final.columns:
        shipment_final[col] = shipment_final[col].fillna(0)

# Telemetry numeric columns
telemetry_numeric_columns = [
    "TelemetryEventCount",
    "ActivePingCount",
    "InactivePingCount",
    "BatteryPowerCount",
    "AvgTemperature_C",
    "MinTemperature_C",
    "MaxTemperature_C",
    "TempExcursionCount",
    "FrozenCargoTempExcursionCount",
    "InvalidLocationCount",
    "FutureTelemetryCount",
    "TelemetryBeforeBookingCount",
    "TelemetryAfterDeliveryCount",
    "ActivePingPct",
    "BatteryPowerPct"
]

for col in telemetry_numeric_columns:
    if col in shipment_final.columns:
        shipment_final[col] = shipment_final[col].fillna(0)

# Boolean columns
boolean_columns = [
    "HasCarbonDataFlag",
    "NoCarbonLogFlag",
    "CO2UnderReportedShipmentFlag",
    "CO2OverReportedShipmentFlag",
    "HasTelemetryFlag",
    "NoTelemetryFlag",
    "ColdCargoFlag",
    "ColdCargoNoTelemetryFlag",
    "TempExcursionFlag",
    "FrozenCargoTempExcursionFlag",
    "LowActivePingFlag"
]

for col in boolean_columns:
    if col in shipment_final.columns:
        shipment_final[col] = shipment_final[col].fillna(False).astype(bool)

In [130]:
final_quality_summary = {
    "Total Shipments": shipment_final["ShipmentID"].nunique(),
    "Total Legs": legs["LegID"].nunique(),

    "Shipments Without Carbon": shipment_final["NoCarbonLogFlag"].sum(),
    "Shipments Without Telemetry": shipment_final["NoTelemetryFlag"].sum(),
    "Cold Cargo Without Telemetry": shipment_final["ColdCargoNoTelemetryFlag"].sum(),

    "CO2 Under-Reported Shipments": shipment_final["CO2UnderReportedShipmentFlag"].sum(),
    "CO2 Over-Reported Shipments": shipment_final["CO2OverReportedShipmentFlag"].sum(),
    "Total CO2 Gap KG": shipment_final["CO2Gap_KG"].sum(),

    "Temperature Excursion Shipments": shipment_final["TempExcursionFlag"].sum(),
    "Frozen Cargo Temperature Excursion Shipments": shipment_final["FrozenCargoTempExcursionFlag"].sum(),

    "Low Active Ping Shipments": shipment_final["LowActivePingFlag"].sum(),

    "Customs Overcharged Shipments": shipment_final["CustomsOverchargedShipmentFlag"].sum(),
    "Late But No Delay Reason": shipment_final["ShipmentLateNoLegDelayReasonFlag"].sum(),
    "Carrier Mode Mismatch Legs": legs["CarrierModeMismatchFlag"].sum()
}

print(pd.Series(final_quality_summary))

Total Shipments                                 1.500000e+04
Total Legs                                      2.703100e+04
Shipments Without Carbon                        0.000000e+00
Shipments Without Telemetry                     8.979000e+03
Cold Cargo Without Telemetry                    0.000000e+00
CO2 Under-Reported Shipments                    1.500000e+04
CO2 Over-Reported Shipments                     0.000000e+00
Total CO2 Gap KG                                1.674431e+08
Temperature Excursion Shipments                 1.043000e+03
Frozen Cargo Temperature Excursion Shipments    3.031000e+03
Low Active Ping Shipments                       0.000000e+00
Customs Overcharged Shipments                   2.763000e+03
Late But No Delay Reason                        9.455000e+03
Carrier Mode Mismatch Legs                      9.675000e+03
dtype: float64


In [131]:
shipment_final.to_csv(r'C:\Users\nooru\Desktop\logistics_dashboard\final_clean_data\shipment_final_cleaned.csv', index=False)
legs.to_csv(r'C:\Users\nooru\Desktop\logistics_dashboard\final_clean_data\shipment_leg_cleaned_final.csv', index=False)
carbon.to_csv(r'C:\Users\nooru\Desktop\logistics_dashboard\final_clean_data\carbon_logs_cleaned_final.csv', index=False)
carbon_shipment_agg.to_csv(r'C:\Users\nooru\Desktop\logistics_dashboard\final_clean_data\carbon_shipment_agg.csv', index=False)
telemetry_shipment_agg.to_csv(r'C:\Users\nooru\Desktop\logistics_dashboard\final_clean_data\telemetry_shipment_agg.csv', index=False)
telemetry_daily_agg.to_csv(r'C:\Users\nooru\Desktop\logistics_dashboard\final_clean_data\telemetry_daily_agg.csv', index=False)

